In [ ]:
!pip install langchain faiss-cpu sentence-transformers transformers torch langchain-huggingface -q
import warnings
from transformers import logging as hf_logging

warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.file_download")

hf_logging.set_verbosity_error()

from langchain_huggingface import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.docstore.document import Document
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline

# Knowledge base

docs = [
    Document(page_content="AI Tutoring Platform: personalized learning with real-time feedback"),
    Document(page_content="VR Classroom: immersive learning experiences with gamified lessons"),
    Document(page_content="Smart Study Planner: tracks learning habits and suggests improvements"),
    Document(page_content="Peer-to-peer learning apps using AI to match students with similar goals"),
    Document(page_content="AI-driven assessment tools for adaptive testing"),
    Document(page_content="AI mentorship platforms to guide students in career decisions"),
    Document(page_content="Language learning apps using AI speech recognition and feedback"),
    Document(page_content="AI homework assistant for collaborative problem-solving"),
    Document(page_content="AI-powered learning analytics dashboards for teachers"),
    Document(page_content="Edutainment apps combining games and AI-driven quizzes"),
    Document(page_content="RAG improves AI answers by retrieving relevant knowledge first then generating context-aware responses"),
    Document(page_content="Generative AI can assist in brainstorming, summarization, and content creation")
]

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)

# High-end free model
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=500)
local_llm = HuggingFacePipeline(pipeline=pipe)

# RAG QA chain (map_reduce)
qa = RetrievalQA.from_chain_type(
    llm=local_llm,
    chain_type="map_reduce",  # generates richer output from multiple docs
    retriever=vectorstore.as_retriever()
)

# Helper function
def ask_rag(query):
    return qa.invoke(query)['result']

# Structured queries
query1 = """You are an AI innovation expert. Using the documents retrieved, generate 3 innovative startup ideas for education.
Format each idea as:
1) Idea Name: short description
2) Idea Name: short description
3) Idea Name: short description"""
print(" Query:", query1)
print("\n💡 AI Response:\n", ask_rag(query1))

query2 = """Explain how RAG improves the creativity and accuracy of AI-generated content, using simple examples."""
print("\n----------------------------\n")
print("Query:", query2)
print("\n💡 AI Response:\n", ask_rag(query2))


 Query: You are an AI innovation expert. Using the documents retrieved, generate 3 innovative startup ideas for education.
Format each idea as:
1) Idea Name: short description
2) Idea Name: short description
3) Idea Name: short description

💡 AI Response:
 Generative AI can assist in brainstorming, summarization, and content creation AI Tutoring Platform: personalized learning with real-time feedback AI mentorship platforms to guide students in career decisions

----------------------------

Query: Explain how RAG improves the creativity and accuracy of AI-generated content, using simple examples.

💡 AI Response:
 RAG improves AI answers by retrieving relevant knowledge first then generating context-aware responses
